# Lossless BPE optimization — Maick Dane Nkou

**Current reference: your uploaded tokenizer, validation score 2.023252 (reported
from your Colab run), 100% reconstruction, no guardrail penalty.** This notebook
remeasures the exact reference on the same validation data as every candidate.
It does not compare against the misleading old base-only score of 1.7440.

Run all in Colab. The bounded search makes **two inexpensive pre-tokenizer-only
trials, then seven full training runs**. It tests punctuation-preserving and
combining-mark-aware boundaries, language weights, and minimum frequency. Every
new vocabulary/merge table is trained from scratch on **train only**. The
reference and its two adapted copies come from your own previous training.

Selection uses the **full official validation score**, not raw fertility. A
candidate must reconstruct every original validation string, emit no UNK, have
no guardrail penalty, and keep at least 5% English/French guardrail headroom.
The reference is retained unless an eligible candidate is strictly better.
Hidden-test improvement is not guaranteed; avoid repeatedly tuning to validation.

The previous lossless training notebook is preserved at Git commit
`2805e5a52b1f9a859e871387ec71b952914f6ab3`. The current reference artifact is pinned
to `07264e0b0a20d3e179f5f4ecad5448505e2dbd1d` and verified by SHA-256.
No new competition score is claimed until the search has actually run.

## 1. Dependencies
The exact tokenizers version is part of the submission contract.

In [ ]:
%pip install -q "tokenizers==0.22.1" "datasets>=4,<5" "PyYAML==6.0.2"
import tokenizers
assert tokenizers.__version__ == "0.22.1", "Restart the runtime after installing dependencies"
print("tokenizers:", tokenizers.__version__)

## 2. Search recipes and safety gates
All variants preserve raw case, Unicode representation and whitespace. They
use a complete 256-byte alphabet, no special tokens and a ByteLevel decoder.

- `bytelevel`: the current standard ByteLevel boundaries.
- `letter_marks`: letters and combining marks stay together (useful for Yoruba).
- `space_word`: punctuation can stay attached to its word. Unlike
  **WhitespaceSplit**, `Split(..., behavior="isolated")` retains all separators.

Boundary changes are hypotheses, not guaranteed improvements. Both adapted
reference copies and newly trained variants must pass full validation. The
strict round-trip gate is stronger than the pinned official reconstruction metric.

In [ ]:
import hashlib
import importlib.util
import json
import math
import random
import shutil
import tempfile
import time
import unicodedata
import urllib.request
from collections import Counter
from pathlib import Path

from tokenizers import Regex, Tokenizer, decoders, models, pre_tokenizers, trainers

LANGUAGES = ("en", "fr", "ha", "sw", "yo", "am")
SCORED_LANGUAGES = ("ha", "sw", "yo", "am")
DATASET_ID = "Similoluwa/african-multilingual-tokenizer-challenge"
DATASET_REVISION = "v1.0.0"
RECIPE = "lossless-bpe-search"
MIN_GUARDRAIL_HEADROOM = 0.05
IMPROVEMENT_EPSILON = 1e-8
BOOST = {"ha": 3, "sw": 3, "yo": 3, "am": 3}
VOCAB_SIZE = 10_000
MIN_FREQUENCY = 5
MAX_FILE_BYTES = 20 * 1024 * 1024
TEAM_SLUG = "maick-dane-nkou"
# Generated data/models/reports stay outside the Git submission directory.
RUN_DIR = Path.cwd() / "artifacts" / RECIPE
RUN_DIR.mkdir(parents=True, exist_ok=True)


# The custom Split retains matched and unmatched text; nothing is removed.
BOUNDARY_PATTERNS = {
    "letter_marks": r" ?[\p{L}\p{M}]+| ?\p{N}+| ?[^\s\p{L}\p{M}\p{N}]+|\s+(?!\S)|\s+",
    "space_word": r" ?\S+|\s+",
}


def make_pre_tokenizer(mode):
    if mode == "bytelevel":
        return pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=True)
    if mode not in BOUNDARY_PATTERNS:
        raise ValueError(f"Unknown boundary mode: {mode}")
    return pre_tokenizers.Sequence([
        pre_tokenizers.Split(Regex(BOUNDARY_PATTERNS[mode]), behavior="isolated"),
        pre_tokenizers.ByteLevel(add_prefix_space=False, use_regex=False),
    ])


def new_tokenizer(mode="bytelevel"):
    tok = Tokenizer(models.BPE())
    tok.pre_tokenizer = make_pre_tokenizer(mode)
    tok.decoder = decoders.ByteLevel()
    return tok


def train_tokenizer(texts, *, vocab_size=VOCAB_SIZE, min_frequency=MIN_FREQUENCY, mode="bytelevel"):
    tok = new_tokenizer(mode)
    trainer = trainers.BpeTrainer(
        vocab_size=vocab_size,
        min_frequency=min_frequency,
        initial_alphabet=sorted(pre_tokenizers.ByteLevel.alphabet()),
        special_tokens=[],
        show_progress=True,
    )
    tok.train_from_iterator(texts, trainer=trainer)
    assert tok.normalizer is None
    assert tok.get_vocab_size(with_added_tokens=True) <= vocab_size
    assert set(pre_tokenizers.ByteLevel.alphabet()) <= set(tok.get_vocab())
    return tok


def corpus_iterator(train_by_lang, boost=None):
    """Stable round-robin; only the train split is passed here."""
    boost = BOOST if boost is None else boost
    iterators = {lang: iter(train_by_lang[lang]) for lang in LANGUAGES}
    active = list(LANGUAGES)
    while active:
        for lang in active.copy():
            try:
                text = next(iterators[lang])
            except StopIteration:
                active.remove(lang)
                continue
            for _ in range(boost.get(lang, 1)):
                yield text  # original string, with all case/whitespace intact


def assert_exact_roundtrip(tok, texts, *, batch_size=512):
    """Fail on any lost character, including boundary whitespace or special text."""
    failures = 0
    first_indices = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encoded = tok.encode_batch(batch, add_special_tokens=False)
        ids = [enc.ids for enc in encoded]
        decoded = tok.decode_batch(ids, skip_special_tokens=False)
        skipped = tok.decode_batch(ids, skip_special_tokens=True)
        for i, (original, restored, restored_skip) in enumerate(zip(batch, decoded, skipped, strict=True)):
            if original != restored or original != restored_skip:
                failures += 1
                if len(first_indices) < 5:
                    first_indices.append(start + i)
    if failures:
        raise ValueError(f"Export blocked: {failures}/{len(texts)} lossy rows; first indices {first_indices}")
    return len(texts)


def assert_submission_ready(report, expected_rows):
    # The official `valid` flag alone does NOT reject lossy models.
    if not report.get("valid"):
        raise ValueError(f"Official validity checks failed: {report.get('errors')}")
    if report.get("rows") != expected_rows:
        raise ValueError("Official checker did not evaluate the complete validation split")
    if (report.get("lossy_rows") != 0 or report.get("reconstruction") != 1.0
            or report.get("reconstruction_penalty") != 0.0):
        raise ValueError("Export blocked: official reconstruction is not 100%")
    if set(report.get("fertility", {})) != set(LANGUAGES):
        raise ValueError("Missing validation language")
    if set(report.get("unknown_rate", {})) != set(LANGUAGES):
        raise ValueError("Missing unknown-token measurements")
    if any(rate != 0.0 for rate in report["unknown_rate"].values()):
        raise ValueError("Export blocked: unknown tokens were emitted")
    for key in ("score", "guardrail_penalty", "reconstruction_penalty"):
        if not math.isfinite(report[key]) or report[key] < 0:
            raise ValueError(f"Invalid metric: {key}")
    # Guardrail overages are allowed by the competition but must be included.
    base = sum(report["penalised"][lang] for lang in SCORED_LANGUAGES) / 4
    expected = base + report["guardrail_penalty"] + report["reconstruction_penalty"]
    if not math.isclose(report["score"], expected, rel_tol=1e-12, abs_tol=1e-12):
        raise ValueError("Score is missing a penalty")

## 3. Regression tests (not competition training)
These small, disposable models test the implementation only. Their vocabulary
and merge table are **never reused** by the final training run. Tests cover all
six languages, case, NFC/NFD, repeated spaces, tabs, line breaks, emoji, literal
special-token strings, and previously unseen Unicode characters. They also
verify that lowercasing, a mismatched decoder, and artificial prefix spaces are rejected.

In [ ]:
SMOKE_TEXTS = [
    "Knowledge grows when it is shared.",
    "Le savoir grandit lorsqu’il est partagé.",
    "Ilimi yana ƙaruwa idan an raba shi.",
    "Maarifa hukua yanaposhirikishwa.",
    "Ìmọ̀ ń pọ̀ sí i nígbà tí a bá pín in.",
    "እውቀት ሲካፈል ያድጋል።",
]
ROUNDTRIP_CASES = SMOKE_TEXTS + [
    "", " ", "   ", "\t\n\r\n", "  Hello  WORLD!\tNext\nline.  ",
    "[UNK] [CLS] [SEP] <0xFF> <s>", "é e\u0301 Ì I\u0300",
    "👩🏿‍💻 🌍 中文 العربية", "a\u00a0b\u2003c\u200bd",
    "\x00\x01\x7f\ufeff\U0010ffff", "don't l’amour — … ።",
]
ROUNDTRIP_CASES += [unicodedata.normalize("NFD", text) for text in SMOKE_TEXTS]
rng = random.Random(41)
scalars = []
while len(scalars) < 256:
    point = rng.randrange(0x110000)
    if not 0xD800 <= point <= 0xDFFF:  # Python surrogates are not Unicode scalar values
        scalars.append(chr(point))
ROUNDTRIP_CASES += ["".join(scalars), " ".join(scalars)]


def expect_rejected(action):
    try:
        action()
    except ValueError:
        return
    raise AssertionError("Regression: a lossy/invalid candidate was accepted")


def run_regression_tests():
    from tokenizers import normalizers
    mini = train_tokenizer(iter(SMOKE_TEXTS), vocab_size=512, min_frequency=1)
    with tempfile.TemporaryDirectory() as tmp:
        path = Path(tmp) / "tokenizer.json"
        mini.save(str(path))
        loaded = Tokenizer.from_file(str(path))
        assert_exact_roundtrip(loaded, ROUNDTRIP_CASES)
        for mutation in ("lowercase", "wrong_decoder", "prefix_space", "whitespace_split"):
            broken = Tokenizer.from_str(loaded.to_str())
            if mutation == "lowercase":
                broken.normalizer = normalizers.Lowercase()
            elif mutation == "wrong_decoder":
                broken.decoder = decoders.ByteFallback()
            elif mutation == "prefix_space":
                broken.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=True)
            else:
                broken.pre_tokenizer = pre_tokenizers.WhitespaceSplit()
            expect_rejected(lambda: assert_exact_roundtrip(broken, ROUNDTRIP_CASES))
    good = {
        "valid": True, "rows": 24_000, "lossy_rows": 0, "reconstruction": 1.0,
        "reconstruction_penalty": 0.0, "guardrail_penalty": 0.2, "score": 2.2,
        "fertility": dict.fromkeys(LANGUAGES, 2.0),
        "unknown_rate": dict.fromkeys(LANGUAGES, 0.0),
        "penalised": dict.fromkeys(LANGUAGES, 2.0),
    }
    assert_submission_ready(good, 24_000)
    for patch in (
        {"lossy_rows": 1}, {"reconstruction": 0.99}, {"reconstruction_penalty": 0.1},
        {"valid": False}, {"rows": 6}, {"score": 2.0}, {"score": float("nan")},
        {"unknown_rate": dict.fromkeys(LANGUAGES, 0.1)},
    ):
        expect_rejected(lambda: assert_submission_ready({**good, **patch}, 24_000))
    print(f"Regression tests passed ({len(ROUNDTRIP_CASES)} exact round-trip cases).")

run_regression_tests()

# Check both newly trained and adapted reference pipelines for every boundary mode.
with tempfile.TemporaryDirectory() as tmp:
    for mode in ("bytelevel", "letter_marks", "space_word"):
        mini = train_tokenizer(SMOKE_TEXTS, vocab_size=512, min_frequency=1, mode=mode)
        path = Path(tmp) / "tokenizer.json"
        mini.save(str(path))
        assert_exact_roundtrip(Tokenizer.from_file(str(path)), ROUNDTRIP_CASES)
        adapted = train_tokenizer(SMOKE_TEXTS, vocab_size=512, min_frequency=1)
        adapted.pre_tokenizer = make_pre_tokenizer(mode)
        assert_exact_roundtrip(adapted, ROUNDTRIP_CASES)
print("All three lossless boundary modes passed serialization and Unicode tests.")


## 4. Pinned official checker — no silent fallback
The old notebook could use a stale `utils.py` and its copied metric omitted penalties.
Here the checker is downloaded from a fixed official Git commit, its SHA-256 is
verified, and that exact module is loaded directly. A failed download stops the run;
an old local `utils.py` is never imported. The pin documents the scoring version;
organizers may update their rules later.

In [ ]:
OFFICIAL_COMMIT = "75578f2400c39b1f8e31ce7e7104b37fbc470d11"
OFFICIAL_UTILS_SHA256 = "1727de34136097eb48addabf90501589bdfefa31c20e201bab53c38f2f7c9688"
OFFICIAL_UTILS_URL = (
    "https://raw.githubusercontent.com/aims-ai-research-foundations/"
    f"airf-multilingual-tokenizer-challenge/{OFFICIAL_COMMIT}/starter/utils.py"
)
with urllib.request.urlopen(OFFICIAL_UTILS_URL, timeout=60) as response:
    helper = response.read()
if hashlib.sha256(helper).hexdigest() != OFFICIAL_UTILS_SHA256:
    raise RuntimeError("Official checker hash mismatch; do not continue")
helper_path = RUN_DIR / "official_utils.py"
helper_path.write_bytes(helper)
spec = importlib.util.spec_from_file_location("pinned_official_utils", helper_path)
official_utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(official_utils)
assert official_utils.REQUIRED_TOKENIZERS_VERSION == "0.22.1"
assert official_utils.RECONSTRUCTION_PENALTY == 3.0
print("Official checker:", OFFICIAL_COMMIT)

## 5. Load only the official public train and validation splits
No external corpus, pretrained tokenizer, or vocabulary is used. The release
must contain 40,000 training and 4,000 validation rows per language. Input text
is never stripped or normalized. There is no reduced-data mode for final export.

In [ ]:
from datasets import load_dataset

train_data, validation_data = load_dataset(
    DATASET_ID, revision=DATASET_REVISION,
    split=["train", "validation"], cache_dir=str(RUN_DIR / "dataset-cache"),
)
train_by_lang = {lang: [] for lang in LANGUAGES}
for row in train_data:
    if row["language"] not in train_by_lang or not isinstance(row["text"], str):
        raise ValueError("Unexpected training row")
    train_by_lang[row["language"]].append(row["text"])
val_rows = [(row["language"], row["text"]) for row in validation_data]
assert {lang: len(texts) for lang, texts in train_by_lang.items()} == dict.fromkeys(LANGUAGES, 40_000)
assert Counter(lang for lang, _ in val_rows) == dict.fromkeys(LANGUAGES, 4_000)
assert all(isinstance(text, str) and text.split() for _, text in val_rows)
validation_texts = [text for _, text in val_rows]
print("Train:", len(train_data), "| Validation:", len(val_rows))

## 6. Candidate evaluation and selection
Every candidate is saved, reloaded and checked against original validation text
before official scoring. Each evaluation report is checkpointed under artifacts.
A 5% guardrail margin is a heuristic, not a guarantee on the hidden test set.
Equal/worse scores never replace the reference. Speed does not drive this search.

In [ ]:
import copy
import csv


def context_headroom(report):
    budget = report["guardrail_budget"]
    if not math.isfinite(budget) or budget <= 0:
        raise ValueError("Invalid guardrail budget")
    return min((budget - report["fertility"][lang]) / budget for lang in ("en", "fr"))


def eligible_for_promotion(report, expected_rows):
    try:
        assert_submission_ready(report, expected_rows)
        margin = context_headroom(report)
    except (ValueError, KeyError, TypeError):
        return False
    return (report["guardrail_penalty"] == 0.0
            and margin >= MIN_GUARDRAIL_HEADROOM - 1e-12)


def select_better(best, candidate, expected_rows):
    if (eligible_for_promotion(candidate["official"], expected_rows)
            and candidate["official"]["score"] < best["official"]["score"] - IMPROVEMENT_EPSILON):
        return candidate
    return best


def reference_config():
    return {"name": "reference-upload", "origin": "uploaded-reference",
            "mode": "bytelevel", "training_mode": "bytelevel",
            "boost": dict(BOOST), "min_frequency": MIN_FREQUENCY,
            "vocab_size": VOCAB_SIZE}


def training_config(name, mode, boost, min_frequency=MIN_FREQUENCY):
    return {"name": name, "origin": "trained-from-scratch", "mode": mode,
            "training_mode": mode, "boost": dict(boost),
            "min_frequency": min_frequency, "vocab_size": VOCAB_SIZE}


def evaluate_candidate(path, config, rows, *, training_seconds=0.0):
    path = Path(path)
    if path.stat().st_size > MAX_FILE_BYTES:
        raise ValueError("File exceeds the 20 MiB limit")
    tok = Tokenizer.from_file(str(path))
    if tok.get_vocab_size(with_added_tokens=True) != config["vocab_size"]:
        raise ValueError("Unexpected vocabulary size")
    assert_exact_roundtrip(tok, ROUNDTRIP_CASES)
    assert_exact_roundtrip(tok, [text for _, text in rows])
    measured = official_utils.profile_submission(path, data=rows, repeats=1, verbose=False)
    assert_submission_ready(measured, len(rows))
    candidate = {"config": copy.deepcopy(config), "path": str(path),
                 "sha256": hashlib.sha256(path.read_bytes()).hexdigest(),
                 "official": measured, "training_seconds": training_seconds,
                 "eligible": eligible_for_promotion(measured, len(rows)),
                 "headroom": context_headroom(measured), "strict_lossy_rows": 0}
    (path.parent / "evaluation.json").write_text(
        json.dumps(candidate, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
    print(f"{config['name']:<32} full={measured['score']:.6f} "
          f"guardrail={measured['guardrail_penalty']:.6f} "
          f"reconstruction={measured['reconstruction']:.1%} "
          f"headroom={candidate['headroom']:.1%} eligible={candidate['eligible']}")
    return candidate


def run_search(train_by_lang, rows, reference, run_dir):
    """Bounded search. Validation rows are only passed to evaluation, never training."""
    run_dir = Path(run_dir)
    best = reference
    history = [reference]
    failures = []

    def checkpoint():
        (run_dir / "search_history.json").write_text(json.dumps(
            {"best": best["config"]["name"], "candidates": history, "failures": failures},
            ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

    def attempt(config, adapt_reference=False):
        nonlocal best
        directory = run_dir / "candidates" / config["name"]
        directory.mkdir(parents=True, exist_ok=True)
        path = directory / "tokenizer.json"
        started = time.perf_counter()
        try:
            if adapt_reference:
                # Only your own uploaded model is reused; no pretrained model.
                original = Path(reference["path"])
                if hashlib.sha256(original.read_bytes()).hexdigest() != reference["sha256"]:
                    raise ValueError("Reference changed during search")
                tok = Tokenizer.from_file(str(original))
                tok.pre_tokenizer = make_pre_tokenizer(config["mode"])
            else:
                tok = train_tokenizer(
                    corpus_iterator(train_by_lang, config["boost"]),
                    vocab_size=config["vocab_size"],
                    min_frequency=config["min_frequency"], mode=config["mode"],
                )
            tok.save(str(path))
            elapsed = time.perf_counter() - started
            del tok
            candidate = evaluate_candidate(path, config, rows, training_seconds=elapsed)
        except ValueError as exc:
            failures.append({"config": config, "error": str(exc)})
            print(f"REJECTED {config['name']}: {exc}")
        else:
            history.append(candidate)
            best = select_better(best, candidate, len(rows))
        checkpoint()

    # Phase A: zero-training boundary trials on your own uploaded model.
    for mode in ("letter_marks", "space_word"):
        config = copy.deepcopy(reference["config"])
        config.update(name=f"adapt-{mode}", origin="adapted-own-reference", mode=mode)
        attempt(config, adapt_reference=True)

    # Phase B: train vocabularies suited to the new boundaries (two full runs).
    for mode in ("letter_marks", "space_word"):
        attempt(training_config(f"train-{mode}-b3", mode, BOOST))

    # Phase C: freeze the best boundary mode before trying three weight choices.
    mode = best["config"]["mode"]
    weights = {
        "b2": dict.fromkeys(SCORED_LANGUAGES, 2),
        "b4": dict.fromkeys(SCORED_LANGUAGES, 4),
        "yo-am4": {"ha": 2, "sw": 2, "yo": 4, "am": 4},
    }
    for label, boost in weights.items():
        attempt(training_config(f"weights-{label}", mode, boost))

    # Phase D: freeze the winning mode/weights; two minimum-frequency trials.
    mode, boost = best["config"]["mode"], dict(best["config"]["boost"])
    for frequency in (2, 10):
        attempt(training_config(f"min-frequency-{frequency}", mode, boost, frequency))
    checkpoint()
    return best, history, failures
# Cheap regression tests of the ranking policy (fabricated metrics, no training).
def test_selection_policy():
    def entry(score, context=1.8):
        fertility = dict.fromkeys(LANGUAGES, score)
        fertility.update(en=context, fr=context)
        metrics = {
            "valid": True, "rows": 24_000, "vocab_size": 10_000,
            "lossy_rows": 0, "reconstruction": 1.0, "reconstruction_penalty": 0.0,
            "guardrail_penalty": 0.0, "guardrail_budget": 1.15 * score,
            "score": score, "fertility": fertility, "penalised": fertility.copy(),
            "unknown_rate": dict.fromkeys(LANGUAGES, 0.0),
        }
        return {"official": metrics}

    baseline = entry(2.023252)
    better = entry(1.95)
    assert select_better(baseline, better, 24_000) is better
    assert select_better(baseline, entry(2.1), 24_000) is baseline
    assert select_better(baseline, entry(2.023252), 24_000) is baseline
    assert select_better(baseline, entry(1.95, context=2.2), 24_000) is baseline
    for patch in (
        {"lossy_rows": 1}, {"reconstruction": 0.99}, {"reconstruction_penalty": 0.1},
        {"rows": 6}, {"score": float("nan")}, {"valid": False},
        {"guardrail_penalty": 0.01, "score": 1.96},
        {"unknown_rate": dict.fromkeys(LANGUAGES, 0.1)},
    ):
        broken = entry(1.95)
        broken["official"].update(patch)
        assert select_better(baseline, broken, 24_000) is baseline
    assert baseline["official"]["score"] == 2.023252
    print("Selection tests passed: no lossy, worse, tied, unbalanced or invalid promotions.")


test_selection_policy()


## 7. Load and remeasure the exact uploaded reference
An immutable commit and SHA-256 prevent accidental evaluation of an old root
`tokenizer.json`. In a checkout, the matching local file can be reused. Otherwise
the notebook downloads your own file from the pinned GitHub commit. A mismatch
stops the run. The old 2.023252 report is informational; comparisons below use
fresh measurements from the same checker/data.

In [ ]:
REFERENCE_COMMIT = "07264e0b0a20d3e179f5f4ecad5448505e2dbd1d"
REFERENCE_SHA256 = "9cc713ef8c779d4b657b10afbb4a1843751e8f69ad419342a95961efebd52edf"
REFERENCE_URL = (
    "https://raw.githubusercontent.com/maick-code/airf-multilingual-tokenizer-challenge/"
    f"{REFERENCE_COMMIT}/submissions/{TEAM_SLUG}/tokenizer.json"
)
local_reference = Path.cwd() / "submissions" / TEAM_SLUG / "tokenizer.json"
if local_reference.is_file() and hashlib.sha256(local_reference.read_bytes()).hexdigest() == REFERENCE_SHA256:
    reference_bytes = local_reference.read_bytes()
else:
    with urllib.request.urlopen(REFERENCE_URL, timeout=60) as response:
        reference_bytes = response.read()
if hashlib.sha256(reference_bytes).hexdigest() != REFERENCE_SHA256:
    raise ValueError("Uploaded reference hash mismatch; do not continue")
reference_dir = RUN_DIR / "reference"
reference_dir.mkdir(parents=True, exist_ok=True)
reference_path = reference_dir / "tokenizer.json"
reference_path.write_bytes(reference_bytes)
reference = evaluate_candidate(reference_path, reference_config(), val_rows)
print("Remeasured reference score:", reference["official"]["score"])
print("Previously reported Colab score: 2.023252 (rounded)")

## 8. Run the bounded search
Seven full-corpus trainings take substantially longer than a single run. Keep
Colab connected. Reports and model files are saved after each candidate. This
cell starts a fresh search when rerun; it does not trust stale cached evaluations.
You can lower the search cost by running only the two adapted-reference trials
in a separate experiment, but never export without rechecking full validation.

In [ ]:
selected, history, failures = run_search(train_by_lang, val_rows, reference, RUN_DIR)
print("\nSelected:", selected["config"]["name"])
print("Reference:", reference["official"]["score"], "| Selected:", selected["official"]["score"])
if selected is reference:
    print("No eligible improvement found. Keeping your uploaded tokenizer unchanged.")
else:
    print(f"Validation improvement: {reference['official']['score'] - selected['official']['score']:.6f}")
print("This is a validation comparison, not a hidden-test result.")

## 9. Recheck the winner and write measured reports
Only after the search finishes is a winner chosen. The selected file is
hash-checked and re-evaluated with three benchmark repeats. Export cannot
silently substitute another model or promote a worse validation score.

In [ ]:
candidate_path = Path(selected["path"])
if hashlib.sha256(candidate_path.read_bytes()).hexdigest() != selected["sha256"]:
    raise ValueError("Selected candidate changed since evaluation")
tokenizer = Tokenizer.from_file(str(candidate_path))
assert_exact_roundtrip(tokenizer, ROUNDTRIP_CASES)
assert_exact_roundtrip(tokenizer, validation_texts)
official_report = official_utils.profile_submission(candidate_path, data=val_rows, repeats=3)
assert_submission_ready(official_report, len(val_rows))
if not math.isclose(official_report["score"], selected["official"]["score"], abs_tol=1e-12, rel_tol=1e-12):
    raise ValueError("Selected candidate score changed; investigate before export")
if official_report["score"] > reference["official"]["score"] + IMPROVEMENT_EPSILON:
    raise ValueError("Export blocked: selected score is worse than the reference")
if selected is not reference and not eligible_for_promotion(official_report, len(val_rows)):
    raise ValueError("Selected candidate no longer passes promotion gates")
base_score = sum(official_report["penalised"][lang] for lang in SCORED_LANGUAGES) / 4
report = {
    "experiment": RECIPE, "selected_config": selected["config"],
    "dataset": {"id": DATASET_ID, "revision": DATASET_REVISION,
                "train_rows": len(train_data), "validation_rows": len(val_rows),
                "train_fingerprint": train_data._fingerprint,
                "validation_fingerprint": validation_data._fingerprint},
    "reference_commit": REFERENCE_COMMIT, "reference_sha256": REFERENCE_SHA256,
    "reference_score": reference["official"]["score"],
    "official_checker_commit": OFFICIAL_COMMIT,
    "official_checker_sha256": OFFICIAL_UTILS_SHA256,
    "tokenizer_sha256": selected["sha256"],
    "tokenizers_version": tokenizers.__version__, "strict_lossy_rows": 0,
    "official": official_report, "failures": failures,
}
report_path = RUN_DIR / "validation_report.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
with (RUN_DIR / "search_results.csv").open("w", newline="", encoding="utf-8") as handle:
    fields = ["candidate", "score", "guardrail_penalty", "reconstruction_penalty", "headroom", "eligible", "sha256"]
    writer = csv.DictWriter(handle, fieldnames=fields)
    writer.writeheader()
    for entry in history:
        metrics = entry["official"]
        writer.writerow({"candidate": entry["config"]["name"], "score": metrics["score"],
                         "guardrail_penalty": metrics["guardrail_penalty"],
                         "reconstruction_penalty": metrics["reconstruction_penalty"],
                         "headroom": entry["headroom"], "eligible": entry["eligible"],
                         "sha256": entry["sha256"]})
print("Measured report:", report_path)
print("Search table:", RUN_DIR / "search_results.csv")

## 10. Export the verified winner (or unchanged reference)
The export includes accurate recipe/provenance and measured results, even if
the winning candidate only changed boundaries on your own previous vocabulary.
Replace the three team files only after reviewing the results; include this
notebook as `notebook.ipynb`. Never submit caches, helpers or experiment models.
An unchanged reference is explicitly labelled, not advertised as an improvement.

In [ ]:
import yaml

assert_submission_ready(official_report, 24_000)
if hashlib.sha256(candidate_path.read_bytes()).hexdigest() != report["tokenizer_sha256"]:
    raise ValueError("Candidate changed since evaluation; rerun the evaluation")
assert_exact_roundtrip(Tokenizer.from_file(str(candidate_path)), validation_texts)
if official_report["score"] > reference["official"]["score"] + IMPROVEMENT_EPSILON:
    raise ValueError("Cannot export a worse candidate")
config = selected["config"]
export_dir = RUN_DIR / "export" / TEAM_SLUG
export_dir.mkdir(parents=True, exist_ok=True)
shutil.copy2(candidate_path, export_dir / "tokenizer.json")
metadata = {
    "team": "Maick Dane Nkou", "members": ["Maick Dane Nkou"],
    "affiliation": "AIMS SOUTH AFRICA",
    "approach": (f"Lossless BPE {official_report['vocab_size']}, {config['mode']} boundaries, "
                 f"no normalization; own official-train model; validation "
                 f"{official_report['score']:.6f}, reconstruction 100%."),
}
(export_dir / "metadata.yml").write_text(yaml.safe_dump(metadata, sort_keys=False), encoding="utf-8")
readme = [
    "# Maick Dane Nkou — lossless BPE", "",
    "Official train only; no pretrained tokenizer or external corpus.", "",
    "## Selected recipe and provenance", "",
    f"- Candidate: `{config['name']}`; origin: `{config['origin']}`",
    f"- Training boundary mode: `{config['training_mode']}`",
    f"- Inference boundary mode: `{config['mode']}`",
    f"- Language repeats: `{json.dumps(config['boost'], sort_keys=True)}` (others x1)",
    f"- BPE, vocabulary {official_report['vocab_size']}, min_frequency={config['min_frequency']}",
    "- tokenizers==0.22.1; no normalizer, special tokens, or post-processor",
    "- Full byte alphabet; ByteLevel decoder; no artificial prefix space",
    f"- Dataset: `{DATASET_ID}` @ `{DATASET_REVISION}`",
    f"- Reference artifact from own training: `{REFERENCE_COMMIT}`",
    "- Uploaded/adapted references reuse only the participant's own train-only vocabulary.",
    "- From-scratch candidates train fresh vocabularies on train; validation is used only for selection.",
    "", "## Measured validation results (24,000 rows; not hidden-test scores)", "",
    "| Language | Tokens/word | UNK rate |", "| --- | ---: | ---: |",
]
for lang in LANGUAGES:
    readme.append(f"| {lang} | {official_report['fertility'][lang]:.6f} | {official_report['unknown_rate'][lang]:.6f} |")
readme += [
    "", f"- Reference full score: {reference['official']['score']:.6f}",
    f"- Selected base score: {base_score:.6f}",
    f"- Guardrail penalty: {official_report['guardrail_penalty']:.6f}",
    f"- Reconstruction penalty: {official_report['reconstruction_penalty']:.6f}",
    f"- **Selected full score: {official_report['score']:.6f}**",
    "- Strict exact reconstruction: 24,000 / 24,000 rows (100%)",
    f"- Official checker commit: `{OFFICIAL_COMMIT}`",
    f"- Tokenizer SHA-256: `{report['tokenizer_sha256']}`",
    "", "## Reproduce", "",
    "Run notebook.ipynb end-to-end in Colab. It loads the pinned own reference, trains",
    "seven candidates on official train only, tries two boundary adaptations, and compares",
    "the full validation score including penalties. It retains the reference if no eligible",
    "candidate improves it. Byte-level boundary definitions are in the notebook.",
    "The original reference training notebook is at commit 2805e5a52b1f9a859e871387ec71b952914f6ab3.",
    "BPE merge ties may differ across retraining runs. Evaluate every generated artifact.", "",
]
(export_dir / "README.md").write_text("\n".join(readme), encoding="utf-8")
print("VERIFIED export:", export_dir)
if selected is reference:
    print("Reference retained; no validation improvement claimed.")
try:
    from google.colab import files
except ImportError:
    print("Not in Colab: retrieve the three files from the export directory.")
else:
    for name in ("tokenizer.json", "metadata.yml", "README.md"):
        files.download(str(export_dir / name))